# Experiment C — One-Class Authenticator: Physical Layer Authentication
**Thesis:** Lightweight Device Authentication in Wireless Communication Using RF Fingerprinting  
**Authors:** Amitha · Tharangi Madushani  
**Supervisor:** Prof. Qinghua Wang

---

## Purpose

Experiments A and B demonstrated that a binary CNN can distinguish DEV01 from DEV02 — a closed-set identification task. This experiment builds the actual **Physical Layer Authentication** system:

> *Given an incoming IQ signal, is it from the enrolled legitimate device — or is it an impostor?*

The authenticator is trained on **one device only**. It learns what that device's hardware fingerprint looks like in raw IQ. At inference it measures how well an incoming signal matches that fingerprint. Signals that match are accepted; signals that do not are rejected — including unknown devices never seen during training.

## Core design principle

The model must learn **only the device-intrinsic hardware fingerprint** — not session patterns, not noise patterns, not modulation patterns. Every design decision in this experiment enforces this:

| Design decision | Pattern eliminated | What remains |
|---|---|---|
| All 4 modulations in training | Modulation-specific IQ patterns | Cross-modulation stable features only |
| Both sessions (S1 + S2) | Session-specific noise floor | Cross-session stable features only |
| Noise augmentation (clean+SNR10+SNR0) | Noise floor memorisation | Signal-intrinsic features only |
| DEV02 never seen during training | Inter-device confusion | DEV01 hardware fingerprint only |

## Architecture: 1D Convolutional Autoencoder

```
Input IQ (128, 2)
    ↓
Encoder: Conv1D blocks → compressed latent vector
    ↓                    (hardware fingerprint representation)
Decoder: Transposed Conv1D → reconstructed IQ (128, 2)
    ↓
Reconstruction error = mean squared error between input and output

Authentication:
    error < threshold → ACCEPT (hardware fingerprint recognised)
    error > threshold → REJECT (unknown hardware)
```

## Evaluation

| Metric | Definition | Data source |
|---|---|---|
| True Accept Rate (TAR) | % of legitimate DEV01 signals correctly accepted | Held-out DEV01 (never seen in training) |
| False Accept Rate (FAR) | % of impostor DEV02 signals incorrectly accepted | Full DEV02 dataset |
| TAR per modulation | TAR evaluated separately per modulation | Confirms hardware fingerprint, not modulation |
| FAR per modulation | FAR evaluated separately per modulation | Confirms rejection is hardware-based |
| SNR robustness | TAR and FAR at clean, SNR20, SNR10, SNR0 | All preprocessed SNR files |

**Threshold criterion:** Operating point where FAR ≤ 5% (SR-2 requirement from thesis)

## Section 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import gc

BASE         = '/content/drive/MyDrive/My Thesis'
PREPROCESSED = os.path.join(BASE, 'Preprocessed')
MODEL_DIR    = os.path.join(BASE, 'Models', 'ExperimentC')
os.makedirs(MODEL_DIR, exist_ok=True)

MODULATIONS  = ['BPSK', 'QPSK', 'GFSK', 'OOK']
SESSIONS     = ['S1', 'S2']
AUG_TAGS     = ['clean', 'SNR10', 'SNR0']
EVAL_SNRS    = [None, 20, 10, 0]
TARGET_FAR   = 0.05          # SR-2: FAR must not exceed 5%

BATCH_SIZE   = 256
MAX_EPOCHS   = 100
LR           = 1e-4
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

experiment_C_results = {}

print('Setup complete.')
print(f'TF          : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU         : {gpus[0].name if gpus else "CPU only"}')
print(f'Output dir  : {MODEL_DIR}')

## Section 2 — Helpers

In [ ]:
def load_npy(session, modulation, snr_tag):
    X = np.load(os.path.join(PREPROCESSED, session,
                             f'{modulation}_{snr_tag}_X.npy'))
    y = np.load(os.path.join(PREPROCESSED, session,
                             f'{modulation}_{snr_tag}_y.npy'))
    return X, y


def load_device_data(device_label, modulations, sessions, snr_tags, verbose=True):
    """
    Load IQ windows for ONE device only.
    device_label: 0 = DEV01, 1 = DEV02
    Returns X array containing only windows from the specified device.
    """
    parts = []
    for mod in modulations:
        for sess in sessions:
            for tag in snr_tags:
                X, y = load_npy(sess, mod, tag)
                X_dev = X[y == device_label]
                parts.append(X_dev)
                if verbose:
                    dev_name = 'DEV01' if device_label == 0 else 'DEV02'
                    print(f'  {sess}/{mod}/{tag} → {dev_name}: {X_dev.shape[0]:,} windows')
                del X, y
    result = np.concatenate(parts)
    del parts
    gc.collect()
    return result


def reconstruction_error(model, X, batch_size=256):
    """
    Compute per-window mean squared reconstruction error.
    Returns array of shape (N,) — one error value per window.
    """
    X_recon = model.predict(X, batch_size=batch_size, verbose=0)
    # MSE per window: mean over 128 samples and 2 channels
    errors = np.mean((X - X_recon) ** 2, axis=(1, 2))
    return errors


def find_threshold_at_far(errors_impostor, target_far=0.05):
    """
    Find the reconstruction error threshold such that
    FAR (fraction of impostor windows accepted) <= target_far.
    Accepts windows with error BELOW threshold.
    """
    # Sort impostor errors descending — threshold is the percentile
    # that lets through at most target_far fraction of impostors
    threshold = np.percentile(errors_impostor, (1 - target_far) * 100)
    return float(threshold)


def compute_tar_far(errors_legit, errors_impostor, threshold):
    tar = float(np.mean(errors_legit    < threshold))
    far = float(np.mean(errors_impostor < threshold))
    return tar, far


def snr_tag(snr):
    return 'clean' if snr is None else f'SNR{snr}'


print('Helpers defined.')

## Section 3 — 1D Convolutional Autoencoder Architecture

In [ ]:
def build_autoencoder(input_shape=(128, 2), latent_dim=32, learning_rate=LR):
    """
    Lightweight 1D Convolutional Autoencoder for RF fingerprint learning.

    Encoder compresses IQ windows into a latent vector representing
    the device hardware fingerprint. Decoder reconstructs the IQ signal.
    Reconstruction error is used as the authentication score.

    Encoder mirrors the classifier from Experiments A and B:
    same Conv1D + AveragePooling structure for consistency.
    """
    # ── Encoder ───────────────────────────────────────────────────────────────
    encoder_input = keras.Input(shape=input_shape, name='iq_input')

    x = layers.Conv1D(32,  kernel_size=7, padding='same',
                      activation='relu', name='enc_conv1')(encoder_input)
    x = layers.AveragePooling1D(pool_size=2, name='enc_pool1')(x)   # → (64, 32)

    x = layers.Conv1D(64,  kernel_size=5, padding='same',
                      activation='relu', name='enc_conv2')(x)
    x = layers.AveragePooling1D(pool_size=2, name='enc_pool2')(x)   # → (32, 64)

    x = layers.Conv1D(128, kernel_size=3, padding='same',
                      activation='relu', name='enc_conv3')(x)
    x = layers.AveragePooling1D(pool_size=2, name='enc_pool3')(x)   # → (16, 128)

    # Flatten to latent vector — hardware fingerprint representation
    x = layers.Flatten(name='flatten')(x)
    latent = layers.Dense(latent_dim, activation='relu',
                          name='latent')(x)                          # → (latent_dim,)

    # ── Decoder ───────────────────────────────────────────────────────────────
    x = layers.Dense(16 * 128, activation='relu',
                     name='dec_dense')(latent)                       # → (2048,)
    x = layers.Reshape((16, 128), name='dec_reshape')(x)            # → (16, 128)

    x = layers.Conv1DTranspose(128, kernel_size=3, strides=2,
                               padding='same', activation='relu',
                               name='dec_conv1')(x)                  # → (32, 128)
    x = layers.Conv1DTranspose(64,  kernel_size=5, strides=2,
                               padding='same', activation='relu',
                               name='dec_conv2')(x)                  # → (64, 64)
    x = layers.Conv1DTranspose(32,  kernel_size=7, strides=2,
                               padding='same', activation='relu',
                               name='dec_conv3')(x)                  # → (128, 32)

    # Output: reconstruct I and Q channels
    decoded = layers.Conv1D(2, kernel_size=1, padding='same',
                            activation='linear',
                            name='iq_output')(x)                     # → (128, 2)

    autoencoder = keras.Model(encoder_input, decoded, name='RFF_Autoencoder')
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse'
    )
    return autoencoder


ae = build_autoencoder()
ae.summary()
print(f'\nTotal parameters: {ae.count_params():,}')

## Section 4 — Load DEV01 Data and Split

**Critical:** Only DEV01 data is loaded for training and validation.  
DEV02 data is never used during training — it only appears during evaluation as the impostor.


In [ ]:
print('Loading DEV01 data (all modulations, S1+S2, augmented)...')
print('DEV02 data is intentionally excluded from training.\n')

X_dev01 = load_device_data(
    device_label=0,           # DEV01 only
    modulations=MODULATIONS,  # all 4 — prevents modulation memorisation
    sessions=SESSIONS,        # S1+S2 — prevents session memorisation
    snr_tags=AUG_TAGS         # clean+SNR10+SNR0 — prevents noise memorisation
)

print(f'\nDEV01 total: {X_dev01.shape[0]:,} windows  shape: {X_dev01.shape}')
print(f'X range    : [{X_dev01.min():.4f}, {X_dev01.max():.4f}]')

# ── Split DEV01: 70% train / 15% val / 15% held-out test ──────────────────────
# Held-out test simulates open-set evaluation:
# these windows were never seen during training
X_tv, X_test_dev01 = train_test_split(
    X_dev01, test_size=0.15, random_state=RANDOM_SEED)
X_train, X_val = train_test_split(
    X_tv, test_size=0.15/(0.70+0.15), random_state=RANDOM_SEED)

del X_dev01, X_tv
gc.collect()

print(f'\nDEV01 split:')
print(f'  Train     : {X_train.shape[0]:,}  (autoencoder training)')
print(f'  Val       : {X_val.shape[0]:,}  (threshold tuning)')
print(f'  Held-out  : {X_test_dev01.shape[0]:,}  (open-set TAR evaluation)')
print(f'\nNote: DEV02 data loaded only in Section 7 for FAR evaluation.')

## Section 5 — Train the Autoencoder

In [ ]:
ckpt_path = os.path.join(MODEL_DIR, 'autoencoder_dev01.keras')

autoencoder = build_autoencoder(learning_rate=LR)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(ckpt_path, monitor='val_loss',
                    save_best_only=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1)
]

print('Training autoencoder on DEV01 data only...')
print('Input = IQ windows, Target = same IQ windows (reconstruction task)')
print()
t0 = time.time()

# Autoencoder: input and target are the SAME — reconstruct the input
history = autoencoder.fit(
    X_train, X_train,           # input = target
    validation_data=(X_val, X_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

elapsed = (time.time() - t0) / 60
best_epoch = np.argmin(history.history['val_loss']) + 1
print(f'\nDone in {elapsed:.1f} min')
print(f'Best epoch    : {best_epoch}')
print(f'Best val loss : {min(history.history["val_loss"]):.6f}')

In [ ]:
# Training curves
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs_ran, hist['loss'],     color='steelblue', linewidth=2, label='Train MSE')
ax.plot(epochs_ran, hist['val_loss'], color='coral',     linewidth=2, label='Val MSE')
ax.axvline(best_epoch, linestyle='--', color='grey', linewidth=1,
           label=f'Best epoch ({best_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Autoencoder Training — DEV01 IQ Reconstruction Loss', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
p = os.path.join(MODEL_DIR, 'C_training_curve.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## Section 6 — Threshold Calibration on Validation Set

The rejection threshold is set using the **validation set** (DEV01 data the model has seen  
in a different context) combined with a portion of DEV02 data used only for threshold tuning.  
The threshold is chosen at the operating point where FAR ≤ 5% (SR-2 requirement).

The full DEV02 test set is held back for the final evaluation in Section 7.


In [ ]:
# Compute reconstruction errors on validation set (DEV01)
print('Computing reconstruction errors on DEV01 val set...')
errors_val_dev01 = reconstruction_error(autoencoder, X_val)

# Load a small portion of DEV02 clean data for threshold calibration only
# Use S1 BPSK clean only — this is NOT the test set
print('Loading DEV02 calibration sample (S1/BPSK/clean only)...')
X_cal, y_cal = load_npy('S1', 'BPSK', 'clean')
X_cal_dev02  = X_cal[y_cal == 1][:5000]   # small sample, threshold calibration only
del X_cal, y_cal
gc.collect()

errors_cal_dev02 = reconstruction_error(autoencoder, X_cal_dev02)
del X_cal_dev02
gc.collect()

# ── ROC curve on validation data ───────────────────────────────────────────────
# Labels: DEV01 = legitimate (1), DEV02 = impostor (0)
all_errors = np.concatenate([errors_val_dev01, errors_cal_dev02])
all_labels = np.concatenate([
    np.ones(len(errors_val_dev01)),
    np.zeros(len(errors_cal_dev02))
])

# For ROC: score = negative reconstruction error
# (lower error = more likely legitimate)
fpr_cal, tpr_cal, thresholds_cal = roc_curve(all_labels, -all_errors)
roc_auc_cal = auc(fpr_cal, tpr_cal)

# Find threshold at FAR <= TARGET_FAR
# FAR = FPR (impostor accepted as legitimate)
valid_idx = np.where(fpr_cal <= TARGET_FAR)[0]
best_idx  = valid_idx[np.argmax(tpr_cal[valid_idx])]
# Convert back: threshold on error space
AUTH_THRESHOLD = -thresholds_cal[best_idx]

tar_cal = tpr_cal[best_idx]
far_cal = fpr_cal[best_idx]

print(f'\nCalibration ROC AUC  : {roc_auc_cal:.4f}')
print(f'Authentication threshold: {AUTH_THRESHOLD:.6f}')
print(f'At this threshold:')
print(f'  TAR (val DEV01) : {tar_cal*100:.2f}%')
print(f'  FAR (cal DEV02) : {far_cal*100:.2f}%  (target ≤ {TARGET_FAR*100:.0f}%)')

In [ ]:
# Reconstruction error distribution plot
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.hist(errors_val_dev01, bins=100, alpha=0.7, color='steelblue',
        label='DEV01 (legitimate)', density=True, edgecolor='none')
ax.hist(errors_cal_dev02, bins=100, alpha=0.7, color='coral',
        label='DEV02 (impostor — calibration sample)', density=True, edgecolor='none')
ax.axvline(AUTH_THRESHOLD, color='black', linestyle='--', linewidth=2,
           label=f'Auth threshold = {AUTH_THRESHOLD:.5f}\n(FAR ≤ {TARGET_FAR*100:.0f}%)')

ax.set_xlabel('Reconstruction MSE (lower = more like trained device)')
ax.set_ylabel('Density')
ax.set_title('Reconstruction Error Distribution\n'
             'DEV01 (legitimate) vs DEV02 (impostor) — Autoencoder trained on DEV01 only',
             fontweight='bold')
ax.legend(fontsize=9.5)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Shade accept/reject regions
ax.axvspan(0, AUTH_THRESHOLD, alpha=0.05, color='green')
ax.axvspan(AUTH_THRESHOLD, ax.get_xlim()[1] if ax.get_xlim()[1] > AUTH_THRESHOLD
           else AUTH_THRESHOLD * 3, alpha=0.05, color='red')
ax.text(AUTH_THRESHOLD * 0.3, ax.get_ylim()[1] * 0.85,
        'ACCEPT', color='green', fontsize=11, fontweight='bold')
ax.text(AUTH_THRESHOLD * 1.1, ax.get_ylim()[1] * 0.85,
        'REJECT', color='red', fontsize=11, fontweight='bold')

plt.tight_layout()
p = os.path.join(MODEL_DIR, 'C_error_distribution.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## Section 7 — Full Evaluation: TAR and FAR per Modulation per SNR

This is the core proof that the autoencoder learned **hardware fingerprint only**,  
not modulation patterns or noise patterns:

- If TAR and FAR are **consistent across modulations** → fingerprint is modulation-independent  
- If TAR and FAR are **consistent across SNR levels** → fingerprint is noise-independent  
- If FAR stays **low across all conditions** → rejection is hardware-based, not coincidental


In [ ]:
print('Full evaluation — TAR and FAR per modulation per SNR')
print('Using authentication threshold from Section 6:', AUTH_THRESHOLD)
print()

eval_results = {}   # {mod: {snr_tag: {tar, far, threshold}}}

for mod in MODULATIONS:
    eval_results[mod] = {}
    for snr in EVAL_SNRS:
        tag = snr_tag(snr)

        # Load full data for this modulation + SNR from S2
        X_full, y_full = load_npy('S2', mod, tag)

        # Split by device
        X_dev01_eval = X_full[y_full == 0]   # legitimate
        X_dev02_eval = X_full[y_full == 1]   # impostor

        # Use held-out DEV01 test windows for TAR
        # (these were never seen during training)
        # For per-modulation evaluation we use S2 data directly
        err_legit   = reconstruction_error(autoencoder, X_dev01_eval)
        err_impostor = reconstruction_error(autoencoder, X_dev02_eval)

        tar, far = compute_tar_far(err_legit, err_impostor, AUTH_THRESHOLD)

        eval_results[mod][tag] = {
            'tar'            : tar,
            'far'            : far,
            'n_legit'        : len(err_legit),
            'n_impostor'     : len(err_impostor),
            'mean_err_legit' : float(np.mean(err_legit)),
            'mean_err_imp'   : float(np.mean(err_impostor)),
        }

        snr_label = 'clean' if snr is None else f'SNR{snr}'
        print(f'  {mod:<6} {snr_label:<8}: '
              f'TAR={tar*100:>6.2f}%  FAR={far*100:>5.2f}%  '
              f'err_legit={np.mean(err_legit):.5f}  '
              f'err_imp={np.mean(err_impostor):.5f}')

        del X_full, y_full, X_dev01_eval, X_dev02_eval
        gc.collect()

experiment_C_results['eval_per_mod_snr'] = eval_results
experiment_C_results['threshold']        = AUTH_THRESHOLD
experiment_C_results['target_far']       = TARGET_FAR

## Section 8 — Proof Table: Hardware Fingerprint, Not Modulation Pattern

In [ ]:
snr_cols   = ['clean', 'SNR20', 'SNR10', 'SNR0']
snr_labels = ['Clean (∞ dB)', 'SNR 20 dB', 'SNR 10 dB', 'SNR 0 dB']

print('TAR (True Accept Rate) — DEV01 correctly accepted:')
print(f'  {"Modulation":<8}  ' + '  '.join(f'{s:>12}' for s in snr_labels))
print('  ' + '-' * 65)
for mod in MODULATIONS:
    row = [eval_results[mod].get(c, {}).get('tar', 0) * 100 for c in snr_cols]
    print(f'  {mod:<8}  ' + '  '.join(f'{v:>10.2f}%' for v in row))

print()
print('FAR (False Accept Rate) — DEV02 incorrectly accepted (lower is better):')
print(f'  {"Modulation":<8}  ' + '  '.join(f'{s:>12}' for s in snr_labels))
print('  ' + '-' * 65)
for mod in MODULATIONS:
    row = [eval_results[mod].get(c, {}).get('far', 0) * 100 for c in snr_cols]
    print(f'  {mod:<8}  ' + '  '.join(f'{v:>10.2f}%' for v in row))

print()
print('Interpretation:')
print('  If TAR is consistent across modulations → fingerprint is NOT modulation-specific')
print('  If FAR is consistent across modulations → rejection is NOT modulation-specific')
print('  Both consistent = hardware fingerprint only')

In [ ]:
# TAR and FAR heatmaps side by side
tar_matrix = np.array([
    [eval_results[mod].get(c, {}).get('tar', 0) * 100 for c in snr_cols]
    for mod in MODULATIONS
])
far_matrix = np.array([
    [eval_results[mod].get(c, {}).get('far', 0) * 100 for c in snr_cols]
    for mod in MODULATIONS
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experiment C — One-Class Authenticator: TAR and FAR\n'
             'Consistent values across modulations = hardware fingerprint only',
             fontsize=12, fontweight='bold')

# TAR heatmap
im1 = ax1.imshow(tar_matrix, cmap='Greens', vmin=50, vmax=100, aspect='auto')
plt.colorbar(im1, ax=ax1, label='TAR (%)')
ax1.set_xticks(range(4))
ax1.set_yticks(range(4))
ax1.set_xticklabels(snr_labels, fontsize=9)
ax1.set_yticklabels(MODULATIONS, fontsize=11)
ax1.set_title('True Accept Rate (DEV01)\nHigher is better', fontweight='bold')
for i in range(4):
    for j in range(4):
        v = tar_matrix[i, j]
        ax1.text(j, i, f'{v:.1f}%', ha='center', va='center',
                 fontsize=10, fontweight='bold',
                 color='white' if v < 75 else 'black')

# FAR heatmap — inverted colormap (low FAR = green)
im2 = ax2.imshow(far_matrix, cmap='RdYlGn_r', vmin=0, vmax=20, aspect='auto')
plt.colorbar(im2, ax=ax2, label='FAR (%)')
ax2.set_xticks(range(4))
ax2.set_yticks(range(4))
ax2.set_xticklabels(snr_labels, fontsize=9)
ax2.set_yticklabels(MODULATIONS, fontsize=11)
ax2.set_title(f'False Accept Rate (DEV02 impostor)\nLower is better  (target ≤ {TARGET_FAR*100:.0f}%)',
              fontweight='bold')
for i in range(4):
    for j in range(4):
        v = far_matrix[i, j]
        ax2.text(j, i, f'{v:.1f}%', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='black')

plt.tight_layout()
p = os.path.join(MODEL_DIR, 'C_tar_far_heatmap.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {p}')

## Section 9 — ROC Curve: Full Test Set

In [ ]:
# Full ROC across all modulations combined — clean condition
print('Computing full ROC on all modulations combined (S2 clean)...')

all_err_legit, all_err_imp = [], []

for mod in MODULATIONS:
    X_full, y_full = load_npy('S2', mod, 'clean')
    err_l = reconstruction_error(autoencoder, X_full[y_full == 0])
    err_i = reconstruction_error(autoencoder, X_full[y_full == 1])
    all_err_legit.append(err_l)
    all_err_imp.append(err_i)
    del X_full, y_full
    gc.collect()

all_err_legit = np.concatenate(all_err_legit)
all_err_imp   = np.concatenate(all_err_imp)

all_errors_roc = np.concatenate([all_err_legit, all_err_imp])
all_labels_roc = np.concatenate([
    np.ones(len(all_err_legit)),
    np.zeros(len(all_err_imp))
])

fpr_roc, tpr_roc, _ = roc_curve(all_labels_roc, -all_errors_roc)
roc_auc_full = auc(fpr_roc, tpr_roc)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_roc * 100, tpr_roc * 100,
        color='steelblue', linewidth=2.5,
        label=f'Autoencoder ROC  (AUC = {roc_auc_full:.4f})')
ax.scatter([far_cal * 100], [tar_cal * 100],
           color='red', s=100, zorder=5,
           label=f'Operating point  (TAR={tar_cal*100:.1f}%, FAR={far_cal*100:.1f}%)')
ax.axvline(TARGET_FAR * 100, linestyle='--', color='grey',
           linewidth=1, label=f'SR-2 limit: FAR ≤ {TARGET_FAR*100:.0f}%')
ax.plot([0, 100], [0, 100], 'k--', linewidth=1, alpha=0.4, label='Random classifier')
ax.set_xlabel('False Accept Rate (%)')
ax.set_ylabel('True Accept Rate (%)')
ax.set_title('One-Class Authenticator ROC\n'
             'All modulations combined, S2 clean', fontweight='bold')
ax.legend(fontsize=9.5)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
p = os.path.join(MODEL_DIR, 'C_roc_curve.png')
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC AUC (all modulations, clean): {roc_auc_full:.4f}')
print(f'Saved: {p}')

experiment_C_results['roc_auc_full'] = float(roc_auc_full)

## Section 10 — Save Results + Final Summary

In [ ]:
experiment_C_results['model_params']   = int(autoencoder.count_params())
experiment_C_results['best_epoch']     = int(best_epoch)
experiment_C_results['best_val_loss']  = float(min(history.history['val_loss']))

json_path = os.path.join(MODEL_DIR, 'experiment_C_results.json')
with open(json_path, 'w') as f:
    json.dump(experiment_C_results, f, indent=2)

print(f'Results saved: {json_path}')
print()
print('═' * 65)
print('  EXPERIMENT C SUMMARY — ONE-CLASS AUTHENTICATOR')
print('═' * 65)
print(f'  Architecture    : 1D Convolutional Autoencoder')
print(f'  Parameters      : {autoencoder.count_params():,}')
print(f'  Trained on      : DEV01 only (all mods, S1+S2, augmented)')
print(f'  Best epoch      : {best_epoch}')
print(f'  Auth threshold  : {AUTH_THRESHOLD:.6f}  (FAR ≤ {TARGET_FAR*100:.0f}%)')
print(f'  Full ROC AUC    : {roc_auc_full:.4f}')
print()
print(f'  TAR / FAR per modulation (clean condition):')
print(f'  {"Modulation":<8}  {"TAR":>8}  {"FAR":>8}')
print('  ' + '-' * 30)
for mod in MODULATIONS:
    tar_ = eval_results[mod]['clean']['tar'] * 100
    far_ = eval_results[mod]['clean']['far'] * 100
    sr2  = '✓' if far_ <= TARGET_FAR * 100 else '✗'
    print(f'  {mod:<8}  {tar_:>6.2f}%  {far_:>6.2f}%  SR-2 {sr2}')
print('═' * 65)
print()
print('Files produced:')
for fn in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, fn)) / 1024
    print(f'  {fn:<45} {size:>7.1f} KB')